# Quantiles and Percentiles

## What is a Quantile?

A quantile is a way to divide sorted data into equal-sized groups. Instead of looking at every single value, a quantile tells you where a specific point sits relative to the rest of the data. For example, if a value is at the 25th percentile, it means 25% of the data falls below it.

Quantiles are useful because they help us understand the spread of data, compare different datasets on the same scale, and spot outliers, values that sit far outside the normal range of quantiles.

## Types of Quantiles

* **Quartiles:** split the data into 4 equal parts.
  * Q1 = 25th percentile
  * Q2 = 50th percentile (this is just the median)
  * Q3 = 75th percentile

* **Quintiles:** split the data into 5 equal parts.

* **Deciles:** split the data into 10 equal parts (D1 to D9, each 10% apart).

* **Percentiles:** split the data into 100 equal parts (P1 to P99, each 1% apart). This is the most fine-grained version, and quartiles, quintiles, and deciles can all be described using percentiles. For example, Q1 is just the 25th percentile.

## A Few Things to Remember

1. The data must be sorted from smallest to largest before finding any quantile.
2. A quantile is really about **location**, it tells you where a point sits in the sorted data, not some separate calculated value.
3. Quantiles are not necessarily actual values that exist in your dataset. If the exact position falls between two data points, it gets interpolated.
4. Every other type (quartiles, quintiles, deciles) can be thought of as specific percentiles. You don't need separate formulas for each, they're all the same idea at different cut points.

## Quartiles

Quartiles split sorted data into 4 equal parts. We'll use the `total_bill` column from the tips dataset to find Q1, Q2 (median), and Q3.

In [2]:
import seaborn as sns

df = sns.load_dataset("tips")
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [3]:
q1 = df['total_bill'].quantile(0.25)
q2 = df['total_bill'].quantile(0.50)
q3 = df['total_bill'].quantile(0.75)

print(f"Q1 (25th percentile): {q1}")
print(f"Q2 (50th percentile / median): {q2}")
print(f"Q3 (75th percentile): {q3}")

Q1 (25th percentile): 13.3475
Q2 (50th percentile / median): 17.795
Q3 (75th percentile): 24.127499999999998


## Explanation

* **13.35** is the value below which 25% of the data lies (Q1).
* **17.80** is the value below which 50% of the data lies (Q2, which is also the median).
* **24.13** is the value below which 75% of the data lies (Q3), meaning the remaining 25% of the data lies above it.


So visually, the four equal parts look like this(we are just manually doing it later in we will do it graphically using a library ):

lowest value ---- Q1 (25%) ----  Q2 (50%) ----    Q3 (75%) ----   highest value

|------ 25% ------|----- 25% -----|----- 25% -----|----- 25% -----|



I'll connect this more concretely in the next topic, the Five-Number Summary, 
where Q1, Q2, and Q3 show up again alongside the min and max.

## Percentile

A percentile is a statistical measure that tells us the percentage of observations in a dataset that fall below a particular value. For example, the 75th percentile is the value below which 75% of the observations in the dataset fall. This is actually the exact same idea as Q3, just described using percentile language instead of quartile language.

### Formula to Locate a Percentile

$$PL = \frac{p}{100}(N + 1)$$

where:
* **PL** = the position (location) of the desired percentile in the sorted data
* **N** = the total number of observations in the dataset
* **p** = the percentile rank we want, meaning which percentile you're trying to find (like 25, 50, or 75), expressed as a plain number, not a decimal
### Important: This formula gives a position, not the final value

PL tells you where to look in your sorted data,not the answer itself. If PL comes out as a whole number, like 5, you just take the 5th value in your sorted data. But if PL comes out as something like 5.75, it means the percentile value sits between the 5th and 6th sorted values, and you have to interpolate between them.

### Manual Calculation Plan

I'll calculate this by hand using the `total_bill` column:
1. Sort all the values from smallest to largest.
2. Note $N$, the total count of values.
3. Plug $N$ and the chosen $p$ into the formula to find PL.
4. Find the value at that position (interpolating if PL isn't a whole number).
5. Compare this manual answer against Python's built-in percentile function.

## Let's do one example
### Problem Statement

Using the `total_bill` column from the tips dataset, manually calculate the **90th percentile** of the bill amounts.

In other words: find the bill amount below which 90% of all customer bills fall.

In [4]:
# the formula is: PL = p / 100 * (N + 1)
# first let's find out N -> Total number of observations or datapoints in the dataset

sorted_bills = df['total_bill'].sort_values().reset_index(drop=True)

# we are going to find out the 90th percentile value
p = 90 
N = sorted_bills.size
N

244

In [5]:
PL = (p / 100) * (N + 1)
PL

220.5

## Evaluation
Here the value 220.5 means the 90th percentile value lies between the values at positions 220 and 221. Let's find out that number.

### Extracting the Actual Value

PL = 220.5 tells us the position, but not the value itself. Since it's not a whole number, the real percentile value sits between the value at position 220 and the value at position 221. We find it using linear interpolation:

$$\text{value} = \text{lower value} + \text{fraction part} \times (\text{upper value} - \text{lower value})$$

Here, the fraction part is just the decimal portion of PL, 0.5 in this case, meaning the answer sits exactly halfway between the two positions.

In [6]:
# position 220 and 221 in human counting = index 219 and 220 in Python as indexes starts at 0
first_place_index = int(PL) - 1 
second_place_index = first_place_index + 1

# 4. Extract values from the pre-sorted series
first_place_value = sorted_bills.iloc[first_place_index]
second_place_value = sorted_bills.iloc[second_place_index]

# 5. Get the fractional/decimal remainder
decimal_part = PL % 1

# 6. Apply interpolation formula
final_number = first_place_value + decimal_part * (second_place_value - first_place_value)
print(f'The 90th percentile value is {final_number}')

The 90th percentile value is 32.54


In [7]:
# Let's use another format easy one

# position 220 and 221 in human counting = index 219 and 220 in Python as indexes starts at 0
lower_val = sorted_bills[219]
upper_val = sorted_bills[220]

fraction = PL % 1   # the decimal part of PL, here it's 0.5

manual_p90 = lower_val + (fraction * (upper_val - lower_val))
print(f"Lower value (position 220): {lower_val}")
print(f"Upper value (position 221): {upper_val}")
print(f"Fraction: {fraction}")
print(f"90th percentile value: {manual_p90}")

Lower value (position 220): 32.4
Upper value (position 221): 32.68
Fraction: 0.5
90th percentile value: 32.54


## Result

So the values at position 220 and 221 are **32.40** and **32.68**. Since the fraction is exactly 0.5, we're just taking the midpoint between these two numbers:

$$32.40 + 0.5 \times (32.68 - 32.40) = 32.54$$

So the 90th percentile of `total_bill` is **32.54**. In plain terms, this means 90% of all customer bills in this dataset are 32.54 dollars or less, and only the top 10% of bills go above that.

Now let's check this against Python's built-in function to see if our manual calculation matches.

In [8]:
builtin_p90 = round(df['total_bill'].quantile(0.90), 2)
print(f"Python's built-in 90th percentile: {builtin_p90}")
print(f"Our manual calculation: {manual_p90}")
print(f"Match: {builtin_p90 == manual_p90}")

Python's built-in 90th percentile: 32.24
Our manual calculation: 32.54
Match: False


## Comparing With Python's Built-in Function

Our manual answer (32.54) and pandas' answer (32.24) don't match exactly, and that's fine. Different tools use slightly different formulas for locating a percentile.

To confirm this, I used numpy with the `weibull` method explicitly:

```python
import numpy as np
np.percentile(df['total_bill'], q=90, method='weibull')
```

This gave **32.54**, exactly matching my manual calculation. So the mismatch wasn't a mistake, pandas' `.quantile()` just uses a different default interpolation method than numpy's `weibull` method, which is the one our formula was based on.

It's the same idea as population variance vs sample variance using different divisors, both are valid, just built on different conventions.

So the takeaway isn't that one number is "right." It's that percentile values can shift a bit depending on the method, so it's worth knowing which one a tool uses before comparing results across libraries.

In [9]:
import numpy as np

np.percentile(df['total_bill'], q=90, method='weibull')

np.float64(32.54)

## Percentile of a Value (The Reverse Direction)

So far we went from a percentile rank to a value. This formula does the opposite: given a specific value, it tells you what percentile rank that value sits at.

$$\text{Percentile Rank} = \frac{X + 0.5Y}{N} \times 100$$

where:
* **X** = number of values strictly below the given value
* **Y** = number of values exactly equal to the given value
* **N** = total number of values in the dataset

**Example:** let's find what percentile rank a bill of exactly \$20.00 sits at in `total_bill`.

In [10]:
value = 20.00

X = (df['total_bill'] < value).sum()
Y = (df['total_bill'] == value).sum()
N = df['total_bill'].size

percentile_rank = ((X + 0.5 * Y) / N) * 100
print(f"X (values below {value}): {X}")
print(f"Y (values equal to {value}): {Y}")
print(f"N (total values): {N}")
print(f"Percentile rank of {value}: {round(percentile_rank)}")

X (values below 20.0): 147
Y (values equal to 20.0): 0
N (total values): 244
Percentile rank of 20.0: 60


In [11]:
#Let's see whether the value 32.54 sits at the 90th percentile.

value = 32.54

X = (df['total_bill'] < value).sum()
Y = (df['total_bill'] == value).sum()
N = df['total_bill'].size

percentile_rank = ((X + 0.5 * Y) / N) * 100
print(f"X (values below {value}): {X}")
print(f"Y (values equal to {value}): {Y}")
print(f"N (total values): {N}")
print(f"Percentile rank of {value}: {round(percentile_rank)}")

X (values below 32.54): 220
Y (values equal to 32.54): 0
N (total values): 244
Percentile rank of 32.54: 90
